# Introduction to Machine Learning: Neural Networks

Before we begin, I thought I would do a bit of additional clarification for course projects: \
https://psuastro410.github.io/project/

For today's lecture, we will be discussing neural networks. Large Language Models (LLMs) are a specific type of neural network, trained on data that consists of written material. In addition, machine learning methods, and in particular neural networks, are used to train data from a variety of different astrophysical disciplines, from galaxies and cosmology, to transients and high-energy astrophysics, as well as planetary and exoplanetary astrophysics. Today, we will discuss the fundamental ideas behind how neural networks work to predict outcomes given input data.

### Accepting and submitting

To accept this assignment on Classroom 50 with your GitHub profile enrolled in the class, click on the link \
https://classroom50.org/PsuAstro410/410astro26/assignments/lecture-24/accept \
and follow the steps the webpage prompts.

To submit this assignment, follow the steps in "First time Classroom 50 setup" and "Accepting and Submitting Assignments" on the course webpage:\
https://psuastro410.github.io/tips/github/

### Resources and Acknowledgements

We have used material from the following sources:\
https://zingale.github.io/computational_astrophysics/machine-learning/machine-learning-types.html \
https://zingale.github.io/computational_astrophysics/machine-learning/neural-net-basics.html \
https://zingale.github.io/computational_astrophysics/machine-learning/neural-net-hidden.html

If interested in learning more, I recommend the course notes below:\
https://introml.mit.edu/notes/

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import random

# Types of Machine Learning

[Machine learning](https://en.wikipedia.org/wiki/Machine_learning)
encompasses a wide range of algorithms.  A basic goal is to learn
about a dataset to allow the algorithm to make predictions.

Some popular methods (see the [Wikipedia page for nice descriptions](https://en.wikipedia.org/wiki/Machine_learning#Models)) include:

* artificial neural networks : these mimic neurons and their
  connections in the brain and are trained on data to set weights
  that then allow us to make predictions.

* [decision trees](
  https://en.wikipedia.org/wiki/Decision_tree_learning) : here
  the underlying data structure is a decision tree that uses branching to classify data.

* random forest regression : this is related to decision
  trees but uses multiple trees and averaging to make predictions.


**We'll focus on artificial neural networks in this lecture**

# Artificial Neural Network Fundamentals

## Neural networks

An [_artifical
neural
network_](https://en.wikipedia.org/wiki/Artificial_neural_network)
mimics the action of neurons in your brain to form connections
between nodes (neurons) that link the input to the output.

Basic idea:

* Create a nonlinear fitting routine with free parameters
* Train the network on data with known inputs and outputs to set the parameters
* Use the trained network on new data to predict the outcome

We can think of a neural network as a map that takes a set of
$N_\mathrm{in}$ parameters and returns a set of $N_\mathrm{out}$
parameters, which we can express this as:

$${\bf z} = {\bf A} \cdot {\bf x}$$

where

$${\bf x} = (x_1, x_2, \ldots, x_{N_\mathrm{in}})$$

are the inputs,

$${\bf z} = (z_1, z_2, \ldots, z_{N_\mathrm{out}})$$

are the outputs, and
${\bf A}$ is an $N_\mathrm{out} \times N_\mathrm{in}$ matrix.

Our goal is to determine the matrix elements of ${\bf A}$.

## Nomenclature

We can visualize a neural network as:

![NN diagram](https://zingale.github.io/computational_astrophysics/_images/nn_fig.png)

* Neural networks are divided into _layers_

  * There is always an _input layer_&mdash;it doesn't do any processing.

  * There is always an _output layer_.

* Within a layer there are neurons or _nodes_.

  * For input, there will be one node for each input variable.  In this figure,
    there are 3 nodes on the input layer.

  * The output layer will have as many nodes are needed to convey the answer
    we are seeking from the network.  In this case, there are 2 nodes on the
    output layer.

* Every node in the first layer connects to every node in the next layer

  * The _weight_ associated with the _connection_ can vary&mdash;these are the matrix elements.

**Note:**
This is called a _dense layer_.  There are alternate types of layers we can explore where the nodes are connected differently.

* In this example, the processing is done in layer 2 (the output)

* When you train the neural network, you are adjusting the weights connecting to the nodes

  * Some connections might have zero weight

  * This mimics nature&mdash;a single neuron can connect to several (or lots) of other neurons.

## Universal approximation theorem

A neural network can be designed to approximate any function, $f(x)$.  For this to work, there must be a source of non-linearity in the network&mdash;this is a result of the [universal approximation theorem](https://en.wikipedia.org/wiki/Universal_approximation_theorem).

We use a nonlinear [_activation function_](https://en.wikipedia.org/wiki/Activation_function) that is applied in a layer.  It has
the form:

$$g({\bf v}) = \left ( \begin{array}{c} g(v_0) \\ g(v_1) \\ \vdots \\ g(v_{n-1}) \end{array} \right )$$

**Note:**
The activation function, $g({\bf v})$ works element-by-element on the vector ${\bf v}$.

Then our neural network has the form: ${\bf z} = g({\bf A} \cdot {\bf x})$

We want to choose a function $g(\xi)$ that is differentiable.  A common choice is the _sigmoid function_:

$$g(\xi) = \frac{1}{1 + e^{-\xi}}$$

In [ ]:
def g(xi):
    return 1/(1 + np.exp(-xi))

In [ ]:
xi = np.linspace(-10, 10, num=100)

fig = plt.figure(1)
plt.plot(xi, g(xi))
plt.show()

**Note:**
There are [many choices for the activation function](https://en.wikipedia.org/wiki/Activation_function) which have
different properties.  Often the choice of activation function will be empirical, by experimenting with the 
performance of the network.

## Loss Function

The idea for the [loss function](https://en.wikipedia.org/wiki/Loss_function) $\mathcal{L}$ is we want to quantify some sort of insentive: the larger $\mathcal{L}$ is, the more you "loose." The simplest form for $\mathcal{L}$, given an expected output $g$ and true answer $y$ is the 0-1 loss function:
$$
\mathcal{L}_{01} = 
\left\{
\begin{array}{cc}
0 & g=y \\
1 & g \ne y
\end{array} \right. .
$$
If we are right, we have no loss, but if we are wrong, we have loss.

We will be using the quadratic loss function

\begin{align}
    \mathcal{L} &= \| g({\bf A} \cdot {\bf x}) - {\bf y} \|^2
\end{align}

There are also [many other loss functions](https://en.wikipedia.org/wiki/Loss_function).

## Fundamental algorithm

We'll consider the case where we have training data---a set of inputs, ${\bf x}^k$,
together with the expected output (answer), ${\bf y}^k$.  These training pairs
allow us to constrain the output of the network and train the weights.

* Training

  * Loop over the $T$ pairs $({\bf x}^k, {\bf y}^k)$ for $k = 1, \ldots, T$

    * Predict the output for ${\bf x}^k$ as:

      $$z_i = g([{\bf A} \cdot {\bf x}^k]_i) = g \left ( \sum_{j=1}^{N_\mathrm{in}} A_{ij} x^k_j \right )$$

    * Compare the prediction ${\bf z}$, to the truth ${\bf y}^k$.

      This is a minimization problem, where we are minimizing
   
      \begin{align}
      \mathcal{L}(A_{ij}) &= \| g({\bf A} \cdot {\bf x}^k) - {\bf y}^k \|^2 \\
                &= \sum_{i=1}^{N_\mathrm{out}} \left [ g\left (\sum_{j=1}^{N_\mathrm{in}} A_{ij} x^k_j \right ) - y^k_i \right ]^2
      \end{align}

    * Update the matrix ${\bf A}$ based on the training pair $({\bf x}^k, {\bf y}^{k})$.

* Using the network with the trained ${\bf A}$, we can now use the network on data we haven't seen before, $\boldsymbol \chi$:

  $$z_i = g([{\bf A} \cdot { {\boldsymbol \chi}}^k]_i) = g \left ( \sum_{j=1}^{N_\mathrm{in}} A_{ij} \chi^k_j \right )$$

There are a lot of details that we still need to figure out involving the training and minimization.
We'll start with minimization: a common minimization technique used with
neural networks is [_gradient descent_](https://en.wikipedia.org/wiki/Gradient_descent).

# Gradient Descent

[Gradient descent](https://en.wikipedia.org/wiki/Gradient_descent) is an algorithm for finding the minimum of a function of multiple variables.  It works on the principle of looking at the local gradient of a function, then moving in the direction where it decreases the fastest.

**Warning:**
Like we saw with MCMC algorithms, there is no guarantee that you arrive at the global minimum instead of a local minimum.

Given a function $f({\bf x})$, where ${\bf x} = (x_0, x_1, \ldots, x_{N-1})$,
the idea is to first compute the derivative:

$$\partial f / \partial {\bf x} = (\partial f/\partial x_0, \partial f/\partial x_1, \ldots, \partial f/\partial x_{N-1})$$

and then move in the opposite direction by some fraction, $\eta$:

$${\bf x} \leftarrow {\bf x} - \eta \frac{\partial f}{\partial {\bf x}}$$

There are different ways to define what $\eta$ should be, but we'll use a fixed value.  We'll call $\eta$ the _learning rate_.

Let's demonstrate this.

## Test function

The [Rosenbrock function](https://en.wikipedia.org/wiki/Rosenbrock_function)
or the _banana function_ is a very difficult problem for minimization.  It
has the form:

$$f(x, y) = (a - x)^2 + b (y - x^2)^2$$

and for $a = 1$ and $b = 100$, the minimimum is at a point $(a, a^2)$.

In [ ]:
def rosenbrock(x0, x1, a, b):
    return (a - x0)**2 + b*(x1 - x0**2)**2

def drosdx(x, a, b):
    x0 = x[0]
    x1 = x[1]
    return np.array([-2.0*(a - x0) - 4.0*b*(x1 - x0**2)*x0,
                     2.0*b*(x1 - x0**2)])

Let's plot the function

In [ ]:
xmin = -2.0
xmax = 2.0
ymin = -1.0
ymax = 3.0
a = 1.0
b = 100.0

N = 256
x = np.linspace(xmin, xmax, N)
y = np.linspace(ymin, ymax, N)

x2d, y2d = np.meshgrid(x, y, indexing="ij")

In [ ]:
fig, ax = plt.subplots()

im = ax.imshow(np.log10(np.transpose(rosenbrock(x2d, y2d, a, b))),
               origin="lower", extent=[xmin, xmax, ymin, ymax])

fig.colorbar(im, ax=ax)
plt.show()

## Implementing gradient descent

Let's start with an initial guess.  We'll keep guessing until the change in the solution is small.

**Note:** our success is very sensitive to our choice of $\eta$.

In [ ]:
x0 = np.array([-1.0, 1.5])

We'll set a tolerance and keep iterating until the change in the solution, `dx` is small

In [ ]:
def do_descent(dfdx, x0, eps=1.e-5, eta=2.e-3, args=None, ax=None):

    # dx will be the change in the solution -- we'll iterate until this
    # is small
    dx = 1.e30
    xp_old = x0.copy()

    if args:
        grad = dfdx(xp_old, *args)
    else:
        grad = dfdx(xp_old)

    while dx > eps:

        xp = xp_old - eta * grad
        
        if ax:
            ax.plot([xp_old[0], xp[0]], [xp_old[1], xp[1]], color="C1")
            
        dx = np.linalg.norm(xp - xp_old)
        
        if args:
            grad_new = dfdx(xp, *args)
        else:
            grad_new = dfdx(xp)
            
        #eta_new = np.abs(np.transpose(xp) @ (grad_new - grad)) / np.linalg.norm(grad_new - grad)**2
        #eta = min(10*eta, eta_new)
        
        grad = grad_new
        
        xp_old[:] = xp

In [ ]:
do_descent(drosdx, x0, args=(a, b), ax=ax)

In [ ]:
fig

## momentum

A variation on gradient descent is to add "momentum"
to the update.  This means that the correct depends
on the past gradients as well as the current one,
via some combination.  This has the effect of reducing
the zig-zag effect that we see in our attempt above.

# Deriving the Learning Correction

For gradient descent, we need to derive the update to the matrix
${\bf A}$ based on training on a set of our data, $({\bf x}^k, {\bf y}^k)$.

**Important:**
The derivation we do here is specific to our choice of loss function, $\mathcal{L}(A_{ij})$
and activation function, $g(\xi)$.

Let's start with our cost function:

$$\mathcal{L}(A_{ij}) = \sum_{i=1}^{N_\mathrm{out}} (z_i - y_i^k)^2 = \sum_{i=1}^{N_\mathrm{out}}
  \Biggl [ g\biggl (\underbrace{\sum_{j=1}^{N_\mathrm{in}} A_{ij} x^k_j}_{\equiv \alpha_i} \biggr ) - y^k_i \Biggr ]^2$$

where we'll refer to the product $\boldsymbol{\alpha} \equiv {\bf
A} \cdot {\bf x}$ to help simplify notation.  This means that ${\bf z} = g(\boldsymbol{\alpha})$.

We can compute the derivative with respect to a single matrix
element, $A_{pq}$ by applying the chain rule:

$$\frac{\partial \mathcal{L}}{\partial A_{pq}} =
  2 \sum_{i=1}^{N_\mathrm{out}} (z_i - y^k_i) \left . \frac{\partial g}{\partial \xi} \right |_{\xi=\alpha_i} \frac{\partial \alpha_i}{\partial A_{pq}}$$


with

$$\frac{\partial \alpha_i}{\partial A_{pq}} = \sum_{j=1}^{N_\mathrm{in}} \frac{\partial A_{ij}}{\partial A_{pq}} x^k_j = \sum_{j=1}^{N_\mathrm{in}} \delta_{ip} \delta_{jq} x^k_j = \delta_{ip} x^k_q$$

and for $g(\xi)$, we will assume the sigmoid function,so

$$\frac{\partial g}{\partial \xi}
  = \frac{\partial}{\partial \xi} \frac{1}{1 + e^{-\xi}}
  =- (1 + e^{-\xi})^{-2} (- e^{-\xi})
  = g(\xi) \frac{e^{-\xi}}{1+ e^{-\xi}} = g(\xi) (1 - g(\xi))$$

which gives us:

\begin{align}
\frac{\partial \mathcal{L}}{\partial A_{pq}} &= 2 \sum_{i=1}^{N_\mathrm{out}}
   (z_i - y^k_i) z_i (1 - z_i) \delta_{ip} x^k_q \\
   &= 2 (z_p - y^k_p) z_p (1- z_p) x^k_q
\end{align}

where we used the fact that the $\delta_{ip}$ means that only a single term contributes to the sum.

**Note:**
Observe that:

* $e_p^k \equiv (z_p - y_p^k)$ is the error on the output layer,
  and the correction is proportional to the error (as we would
  expect).

* The $k$ superscripts here remind us that this is the result of
  only a single pair of data from the training set.

Now ${\bf z}$ and ${\bf y}^k$ are all vectors of size $N_\mathrm{out} \times 1$ and ${\bf x}^k$ is a vector of size $N_\mathrm{in} \times 1$, so we can write this expression for the matrix as a whole as:

$$\frac{\partial \mathcal{L}}{\partial {\bf A}} = 2 ({\bf z} - {\bf y}^k) \circ {\bf z} \circ (1 - {\bf z}) \cdot ({\bf x}^k)^\intercal$$

where the operator $\circ$ represents _element-by-element_ multiplication (the [Hadamard product](https://en.wikipedia.org/wiki/Hadamard_product_(matrices))).

## Performing the update

We could do the update like we just saw with our gradient descent
example: take a single data point, $({\bf x}^k, {\bf y}^k)$ and
do the full minimization, continually estimating the correction,
$\partial \mathcal{L}/\partial {\bf A}$ and updating ${\bf A}$ until we
reach a minimum.  The problem with this is that $({\bf x}^k, {\bf y}^k)$ is only one point in our training data, and there is no
guarantee that if we minimize completely with point $k$ that we will
also be a minimum with point $k+1$.

Instead we take multiple passes through the training data (called _epochs_) and apply only a single push in the direction that gradient
descent suggests, scaled by a _learning rate_ $\eta$.

The overall minimization appears as:

* Loop over epochs

  * Loop over the training data, $\{ ({\bf x}^0, {\bf y}^0), ({\bf x}^1, {\bf y}^1), \ldots \}$.  We'll refer to the current training
    pair as $({\bf x}^k, {\bf y}^k)$

    * Propagate ${\bf x}^k$ through the network, getting the output
      ${\bf z} = g({\bf A} \cdot {\bf x}^k)$

    * Compute the error on the output layer, ${\bf e}^k = {\bf z} - {\bf y}^k$

    * Update the matrix ${\bf A}$ according to:

      $${\bf A} \leftarrow {\bf A} - 2 \,\eta\, {\bf e}^k \circ {\bf z} \circ (1 - {\bf z}) \cdot ({\bf x}^k)^\intercal$$


# A First Example

Let's do a simple example: given a sequence of numbers as input, we want the last number in the sequence to be the output.  Let's see if we can train a network to recognize this.

We'll create a `ModelData` class that generates our data and also has a method `interpret_result()` that allows us to pass in the result of the network and tell us
which of the number it predicts.  This is needed since the network will not know
that we are dealing with integers, so we'll need to round it.

The important members are:

* `ModelData.x` : the input
* `ModelData.y` : the output

Anticipating the need to scale our data later, we'll also create a `ModelData.x_scaled`
which can provide a scaling to the input and `ModelData.y_scaled` which will be the scaled
output.

In [ ]:
class ModelData:
    """this is the model data for our "last number" training set.  We
    produce input of length N consisting of the numbers 0-9, and we set
    the output to be simply the last element of the input vector

    """
    def __init__(self, N=10):
        self.N = N

        # our model input data
        rng = np.random.default_rng()
        self.x = rng.integers(0, high=10, size=N)
        self.x_scaled = self.x
        
        # our scaled model output data
        self.y = np.array([self.x[-1]])
        self.y_scaled = np.array([self.x_scaled[-1]])
        
    def interpret_result(self, out):
        """take the network output and return a the number from the allowed
        sequence we are closest to

        """
        return max(0, min(9, int(np.round(out, decimals=0)[0])))

Here's what our data looks like:

In [ ]:
model = ModelData()
model.x

In [ ]:
model.y

Now we write our network.  We'll make it take the name of the class
that will create the data so we can reuse this network with different
variations.

Some implementation details:

* The network will only deal with `model.x_scaled` and `model.y_scaled`.
* We need to initialize the matrix ${\bf A}$&mdash;we'll use Gaussian random numbers,
  centered on 0 with a width set to $\sqrt{N_\mathrm{in}}$.  This seems to be a common
  choice.
* We'll loop over the data in the training set in a random order (randomizing each epoch).

In [ ]:
class NeuralNetwork:

    def __init__(self, num_training_unique=100, data_class=None):
        
        self.num_training_unique = num_training_unique

        self.train_set = []
        for _ in range(self.num_training_unique):
            self.train_set.append(data_class())

        # initialize our matrix with Gaussian normal random numbers
        # we get the size from the length of the input and output
        model = self.train_set[0]
        self.N_out = len(model.y_scaled)
        self.N_in = len(model.x_scaled)

        rng = np.random.default_rng()
        self.A = rng.normal(0.0, 1.0/np.sqrt(self.N_in),
                           (self.N_out, self.N_in))

    def g(self, xi):
        """our sigmoid function"""
        return 1.0/(1.0 + np.exp(-xi))

    def train(self, n_epochs=10, eta=0.2):
        """Do the minimization for the training"""

        # train
        for _ in range(n_epochs):
            random.shuffle(self.train_set)
            for model in self.train_set:

                # gradient descent -- just a single improvement.  eta
                # here is our learning rate

                # make these column vectors
                x = model.x_scaled.reshape(self.N_in, 1)
                y = model.y_scaled.reshape(self.N_out, 1)

                b = self.A @ x
                z = self.g(b)

                self.A[:,:] += -eta * 2 * (z - y) * z * (1 - z) @ x.T

    def predict(self, model):
        """predict the outcome using our trained matrix A """
        z = self.g(self.A @ model.x_scaled)
        return model.interpret_result(z)
    
    def check_accuracy(self):
        """use the trained network on the training data and return
        the fraction we get correct"""
        
        n_right = 0
        for model in self.train_set:
            y_nn = self.predict(model)
            if y_nn == model.y:
                n_right += 1
        return n_right / len(self.train_set)

Let's create the network and train it.

In [ ]:
nn = NeuralNetwork(num_training_unique=1000, data_class=ModelData)
nn.train(n_epochs=100)

We can ask the network how well it does on the data its already seen

In [ ]:
frac = nn.check_accuracy()
print(f"fraction correct: {frac}")

All that training, and it is only about 10% accurate!  Now let's check it on data its never seen

In [ ]:
err = []
npts = 1000
n_right = 0
for k in range(npts):
    model = ModelData()
    y_nn = nn.predict(model)
    if y_nn == model.y:
        n_right += 1
    err.append(abs(y_nn - model.y))
    
print(f"fraction correct: {n_right / npts}")

Clearly we are not doing that great.  We are getting only ~10% right, which is basically random guessing.  Let's look at a single attempt

In [ ]:
model = ModelData()
model.x

In [ ]:
model.y

Here's what the network predicts

In [ ]:
nn.predict(model)

And here's the prediction before calling the activation function

In [ ]:
nn.A @ model.x_scaled

Part of the problem here is that the network returns a really large number, and the sigmoid function works best when the prediction is in the region where it varies the fastest, $\xi \sim [-1, 1]$

In [ ]:
nn.g(nn.A @ model.x_scaled)

Basically whatever we feed it, the signmoid will return 1 when the numbers are this large.
We need to scale the data.

## Scaled Data

Let's try again, but this time, let's scale the output that we train to by 10 so it falls within $[0.05, 0.95]$.  We choose to add a small offset, $0.05$ to prevent any of the inputs from being
$0$, which will simply cancel out any weight they multiply.

In [ ]:
class ModelDataScaled:
    """this is the model data for our "last number" training set.  We
    produce input of length N consisting of the numbers 0-9, and we set
    the output to be simply the last element of the input vector.  This
    version scales the data to lie between [0, 1]

    """    
    def __init__(self, N=10):
        self.N = N
        self.offset = 0.05
        self.scale_factor = 10
        
        # our model input data
        rng = np.random.default_rng()
        self.x = rng.integers(0, high=10, size=N)
        self.x_scaled = self.x / self.scale_factor + self.offset

        # our scaled model output data
        self.y = np.array([self.x[-1]])
        self.y_scaled = np.array([self.x_scaled[-1]])
                                
    def interpret_result(self, out):
        """take the network output and undo the scaling and round it.

        """
        return max(0, min(9, int(self.scale_factor *
                                 np.round(out / self.offset)[0] * self.offset)))

In [ ]:
nn = NeuralNetwork(num_training_unique=1000, data_class=ModelDataScaled)
nn.train(n_epochs=100)

In [ ]:
nn.check_accuracy()

In [ ]:
err = []
npts = 1000
n_right = 0
for k in range(npts):
    model = ModelDataScaled()
    y_nn = nn.predict(model)
    if y_nn == model.y:
        n_right += 1
    err.append(abs(y_nn - model.y))
    
print(f"fraction correct: {n_right / npts}")

We seem to do a lot better now.

Take a look at the trained matrix:

In [ ]:
nn.A

Notice that by far the largest element (in magnitude) is the last one.  This makes sense, since
we want ${\bf A} \cdot {\bf x}$ to choose the last value in ${\bf x}$.

## Categorical Data

We'll finish this out by looking at a different way to represent the data.
In the previous attempts, the network did not know that it was supposed to predict
an integer from 0-9 and instead gives a real number that we round to the nearest
integer in that range.

We can instead treat the data as categorical, restricting it to take on only
those values.

Let's create the class first and then we'll see what the output looks like.

In [ ]:
class ModelDataCategorical:
    """this is the model data for our "last number" training set.  We
    produce input of length N, consisting of numbers 0-9 and store
    the result in a 10-element array as categorical data.

    """
    def __init__(self, N=10):
        self.N = N
        
        # our model input data
        rng = np.random.default_rng()
        self.x = rng.integers(0, high=10, size=N)
        self.x_scaled = self.x / 10 + 0.05
        
        # our scaled model output data
        self.y = np.array([self.x[-1]])
        self.y_scaled = np.zeros(10) + 0.01
        self.y_scaled[self.x[-1]] = 0.99
        
    def interpret_result(self, out):
        """take the network output and return the number we predict"""
        return np.argmax(out)

This is categorical data&mdash;the answer we are training on is a 10 element array with
a "1" in the slot corresponding to the correct integer (we actually use 0.99).

In [ ]:
model = ModelDataCategorical()
model.x

In [ ]:
model.y_scaled

In [ ]:
model.y

We see that `model.y_scaled[model.y] = 0.99`.  We will pick as the correct answer
the array index that has the highest value.  

One additional benefit of this
is that our matrix $A$ will now be $10\times 10$, so there are more weights to train.

In [ ]:
nn = NeuralNetwork(num_training_unique=1000, data_class=ModelDataCategorical)
nn.train(n_epochs=100)
nn.check_accuracy()

In [ ]:
err = []
npts = 1000
n_right = 0
for k in range(npts):
    model = ModelDataCategorical()
    y_nn = nn.predict(model)
    if y_nn == model.y:
        n_right += 1
    err.append(abs(y_nn - model.y))
    
print(f"fraction correct: {n_right / npts}")

This network seems to do well&mdash;we can get ~ 1/3rd of the
new data correct.  It helps that there are a lot more connections
that we can train.  We note however that the sigmoid function is not the
best choice for categorical data.

# Exercises (10 pt)

**Students who submit exercises before the end of class get an extra 2 pts of extra credit.**

## Exercise 1 - Muddiest Moment (4 pt)

What was the most confusing part of today's lecture?

**Answer here**


## Exercise 2 - Exploring prediction accuracy, changing model parameters and training dataset size (6 pt)

In these in-class examples, we explore how the neural network accuracy depends on the parameters which we train the data. For concreteness, lets use the Categorical Data for the model. When changing these parameters, keep all other parameters fixed.

### Exercise 2(a) - Dependence on training data size (2 pt)

Change the `num_treaining_unique` parameter to `10**4`, and calculate the fraction correct after running the neural network to predict new numbers.

In [ ]:
## Answer here


### Exercise 2(b) - Dependence on epoch number (2 pt)

Change the `n_epoch` parameter to `10**3`,  and calculate the fraction of correctly predicted new numbers.

In [ ]:
## Answer here


### Exercise 2(c) - Learning rate value $\eta$.

Change the learning rate `eta` to a value of `0.5`, rather than its default of `0.2`, and calculate the fraction of correctly predicted new numbers.

In [ ]:
## Answer here
